Le rôle de ce notebook est de générer des données de fraudes à partir du DAG causal généré

In [1]:
import pandas as pd
data = pd.read_csv("../Fraud Detection Dataset.csv")
data.head()


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
0,T1,4174,1292.76,ATM Withdrawal,16.0,Tablet,San Francisco,0,119,13,Debit Card,0
1,T2,4507,1554.58,ATM Withdrawal,13.0,Mobile,New York,4,79,3,Credit Card,0
2,T3,1860,2395.02,ATM Withdrawal,NaN,Mobile,NaN,3,115,9,NaN,0
3,T4,2294,100.10,Bill Payment,15.0,Desktop,Chicago,4,3,4,UPI,0
4,T5,2130,1490.50,POS Payment,19.0,Mobile,San Francisco,2,57,7,Credit Card,0


In [9]:
import os
import json
import re
import time
import urllib3
import requests

# ─────────────────────────────────────────────
# 1. CONFIGURATION API
# ─────────────────────────────────────────────

OLLAMA_API_URL = "https://ollama-api.lab.groupe-genes.fr/api/generate"
MODEL_NAME = "mistral"

SKIP_SSL_VERIFY = True
MAX_RETRIES_LOAD = 2
RETRY_DELAY_SECONDS = 2

if SKIP_SSL_VERIFY:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ─────────────────────────────────────────────
# 2. CHARGEMENT ET FORMATAGE DU GRAPHE CAUSAL
# ─────────────────────────────────────────────

def _normalize_edges(edges_raw: list) -> list:
    FROM_KEYS = ["from", "de", "source", "cause",  "parent", "src", "origine"]
    TO_KEYS   = ["to",   "vers", "target", "cible", "effet",  "child", "dst"]
    JUST_KEYS = ["justification", "justification_experte", "raison", "label", "reason"]

    normalized = []
    for e in edges_raw:
        if isinstance(e, dict):
            src = next((e[k] for k in FROM_KEYS if k in e), None)
            tgt = next((e[k] for k in TO_KEYS   if k in e), None)
            jst = next((e[k] for k in JUST_KEYS if k in e), "")
            if src and tgt:
                normalized.append({
                    "from":          src,
                    "to":            tgt,
                    "justification": jst,
                    # on conserve aussi les méta-données de confiance si présentes
                    "confiance":     e.get("confiance_pct", e.get("confiance", ""))
                })
        elif isinstance(e, (list, tuple)) and len(e) >= 2:
            normalized.append({"from": e[0], "to": e[1], "justification": ""})

    return normalized

def load_causal_graph(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    NODE_KEYS = ["nodes", "noeuds", "nœuds", "vertices", "variables", "features", "columns"]
    EDGE_KEYS = ["edges", "liens", "links", "causal_links", "arcs", "arrows", "relations"]

    node_key = next((k for k in NODE_KEYS if k in raw), None)
    edge_key  = next((k for k in EDGE_KEYS if k in raw), None)

    if node_key and edge_key:
        edges = _normalize_edges(raw[edge_key])
        return {
            "nodes":      raw[node_key],
            "edges":      edges,
            "commentaire": raw.get("commentaire_expert", "")
        }

    raise ValueError(
        f"Format non reconnu. Clés trouvées : {list(raw.keys())}"
    )


def format_graph_for_prompt(graph: dict) -> str:
    lines = ["Variables du dataset :", ""]
    for node in graph["nodes"]:
        lines.append(f"  - {node}")

    lines += ["", "Relations causales (A → B : justification [confiance]) :", ""]
    for edge in graph["edges"]:
        confiance = f" [confiance : {edge['confiance']}]" if edge.get("confiance") else ""
        lines.append(
            f"  - {edge['from']} → {edge['to']} : {edge['justification']}{confiance}"
        )

    # Nœuds puits = variable(s) cible(s)
    sources = {e["from"] for e in graph["edges"]}
    targets = {e["to"]   for e in graph["edges"]}
    sink_nodes = targets - sources
    if sink_nodes:
        lines += ["", f"Variable(s) cible(s) : {', '.join(sink_nodes)}"]

    # Commentaire expert si présent
    if graph.get("commentaire"):
        lines += ["", f"Contexte expert : {graph['commentaire'][:300]}"]

    return "\n".join(lines)


def get_feature_list(graph: dict) -> list[str]:
    """Retourne la liste des variables hors cibles (features à générer)."""
    targets = {e["to"] for e in graph["edges"]} - {e["from"] for e in graph["edges"]}
    return [n for n in graph["nodes"] if n not in targets]


# ─────────────────────────────────────────────
# 3. CONSTRUCTION DU PROMPT
# ─────────────────────────────────────────────

def build_prompt(graph: dict, n_samples: int = 5) -> str:
    graph_description = format_graph_for_prompt(graph)
    features = get_feature_list(graph)
    fields_str = ", ".join(features + ["Fraudulent"])

    return f"""
Tu es un expert en détection de fraude bancaire.

Le graphe causal suivant représente les relations causales entre les variables
du dataset de détection de fraude. Chaque flèche indique qu'une variable influence
directement une autre.

{graph_description}

Ta tâche est de GÉNÉRER des cas de fraude réalistes en respectant scrupuleusement
ces relations causales :
- Les valeurs générées doivent être cohérentes avec les dépendances causales décrites
- Chaque transaction doit contenir des signaux suspects typiques d'une fraude
- Respecte les chemins causaux (ex: si Previous_Fraudulent_Transactions est élevé,
  alors Number_of_Transactions_Last_24H et Transaction_Amount doivent l'être aussi)
- Raisonne étape par étape mais ne montre PAS ton raisonnement
- Réponds UNIQUEMENT avec un objet JSON valide, sans texte autour, sans markdown

Champs attendus pour chaque transaction : {fields_str}

Voici des exemples de transactions FRAUDULEUSES :

Exemple 1 :
{{
  "User_ID": "USR_8821",
  "Account_Age": 1,
  "Previous_Fraudulent_Transactions": 3,
  "Number_of_Transactions_Last_24H": 12,
  "Transaction_Amount": 2450,
  "Transaction_Type": "Bill Payment",
  "Device_Used": "Tablet",
  "Location": "Boston",
  "Time_of_Transaction": 2,
  "Fraudulent": 1
}}

Exemple 2 :
{{
  "User_ID": "USR_3345",
  "Account_Age": 2,
  "Previous_Fraudulent_Transactions": 5,
  "Number_of_Transactions_Last_24H": 9,
  "Transaction_Amount": 1800,
  "Transaction_Type": "Online Transfer",
  "Device_Used": "Tablet",
  "Location": "Boston",
  "Time_of_Transaction": 23,
  "Fraudulent": 1
}}

Maintenant, génère {n_samples} nouvelles transactions frauduleuses réalistes.
Réponds avec un JSON valide contenant une clé "transactions" qui est une liste
de {n_samples} objets ayant exactement les mêmes champs que les exemples.
"""


# ─────────────────────────────────────────────
# 4. APPEL API OLLAMA AVEC RETRY
# ─────────────────────────────────────────────

def call_ollama(prompt: str) -> str | None:
    """Appelle l'API Ollama et retourne le texte généré."""
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.5,
            "num_predict": 1200,
        }
    }

    for attempt in range(1, MAX_RETRIES_LOAD + 1):
        print(f"🔄 Tentative {attempt}/{MAX_RETRIES_LOAD} — appel à {MODEL_NAME}...")
        try:
            response = requests.post(
                OLLAMA_API_URL,
                json=payload,
                verify=not SKIP_SSL_VERIFY,
                timeout=120
            )
            response.raise_for_status()
            return response.json().get("response", "")

        except requests.exceptions.HTTPError as e:
            if response.status_code in (503, 429):
                print(f"⏳ Modèle en chargement (HTTP {response.status_code}), "
                      f"nouvel essai dans {RETRY_DELAY_SECONDS}s...")
                time.sleep(RETRY_DELAY_SECONDS)
            else:
                print(f"❌ Erreur HTTP : {e}")
                break

        except requests.exceptions.RequestException as e:
            print(f"❌ Erreur réseau : {e}")
            break

    return None


# ─────────────────────────────────────────────
# 5. EXTRACTION ET PARSING DU JSON
# ─────────────────────────────────────────────

def extract_json(text: str) -> dict | list | None:
    """
    Tente plusieurs stratégies pour extraire un JSON valide
    depuis la réponse brute du modèle.
    """
    # Stratégie 1 : bloc ```json ... ```
    match = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # Stratégie 2 : premier objet { ... } ou tableau [ ... ] trouvé
    for start_char, end_char in [('{', '}'), ('[', ']')]:
        start = text.find(start_char)
        end = text.rfind(end_char) + 1
        if start != -1 and end > start:
            try:
                return json.loads(text[start:end])
            except json.JSONDecodeError:
                pass

    return None


# ─────────────────────────────────────────────
# 6. VALIDATION DES TRANSACTIONS GÉNÉRÉES
# ─────────────────────────────────────────────

def validate_transactions(transactions: list, graph: dict) -> list:
    """
    Vérifie que chaque transaction contient tous les champs attendus
    et que Fraudulent vaut bien 1.
    """
    expected_fields = set(graph["nodes"])
    valid, invalid = [], []

    for i, tx in enumerate(transactions):
        missing = expected_fields - set(tx.keys())
        if missing:
            print(f"  ⚠️  Transaction {i+1} — champs manquants : {missing}")
            invalid.append(tx)
        elif tx.get("Fraudulent") != 1:
            print(f"  ⚠️  Transaction {i+1} — Fraudulent != 1 ({tx.get('Fraudulent')})")
            invalid.append(tx)
        else:
            valid.append(tx)

    return valid


# ─────────────────────────────────────────────
# 7. MAIN
# ─────────────────────────────────────────────

if __name__ == "__main__":
    N_SAMPLES = 5

    # Chargement du graphe
    graph = load_causal_graph("dag_causal_fraude.json")
    print(f"   {len(graph['nodes'])} nœuds, {len(graph['edges'])} arêtes chargés.\n")

    # Affichage de la description générée
    print("=== Description du graphe injectée dans le prompt ===")
    print(format_graph_for_prompt(graph))
    print()

    # Construction et envoi du prompt
    prompt = build_prompt(graph, n_samples=N_SAMPLES)
    print(f"🚀 Génération de {N_SAMPLES} transactions frauduleuses avec {MODEL_NAME}\n", flush=True)

    raw_text = call_ollama(prompt)
    if raw_text is None:
        print("❌ Impossible d'obtenir une réponse du modèle.")
        exit(1)

    print("✅ Réponse reçue\n")
    print("=== Réponse brute ===")
    print(raw_text, flush=True)

    # Parsing
    parsed = extract_json(raw_text)
    if parsed is None:
        print("\n⚠️  Aucun JSON valide trouvé dans la réponse.")
        exit(1)

    transactions = parsed if isinstance(parsed, list) else parsed.get("transactions", [])

    # Validation
    print(f"\n=== Validation des {len(transactions)} transaction(s) ===", flush=True)
    valid_transactions = validate_transactions(transactions, graph)
    print(f"   ✅ {len(valid_transactions)} valide(s) / {len(transactions)} générée(s)\n", flush=True)

    # Affichage final
    print("=== Transactions valides ===")
    for i, tx in enumerate(valid_transactions, 1):
        print(f"\n  Transaction {i} :")
        for k, v in tx.items():
            print(f"    {k}: {v}")

    # Sauvegarde optionnelle en JSON
    output_path = "transactions_frauduleuses.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump({"transactions": valid_transactions}, f, indent=2, ensure_ascii=False)
    print(f"\n💾 Résultats sauvegardés dans : {output_path}")

   8 nœuds, 11 arêtes chargés.

=== Description du graphe injectée dans le prompt ===
Variables du dataset :

  - Account_Age
  - Location
  - Transaction_Type
  - Device_Used
  - Number_of_Transactions_Last_24H
  - Payment_Method
  - Time_of_Transaction
  - Fraudulent

Relations causales (A → B : justification [confiance]) :

  - Location → Fraudulent : Some locations have higher rates of fraud [confiance : 34.6%]
  - Transaction_Type → Fraudulent : Online purchases, POS payments, bill payments, bank transfers, and ATM withdrawals can potentially involve fraud [confiance : 12.7%]
  - Device_Used → Fraudulent : Certain devices might be more susceptible to fraud [confiance : 12.7%]
  - Payment_Method → Fraudulent : Different payment methods may involve varying levels of risk for fraud [confiance : 12.7%]
  - Location → Transaction_Type : Different locations might prefer different transaction types [confiance : 7.7%]
  - Location → Device_Used : Different devices might be preferred in ce

🔄 Tentative 1/2 — appel à mistral...
✅ Réponse reçue

=== Réponse brute ===
 {
  "transactions": [
    {
      "User_ID": "USR_9901",
      "Account_Age": 3,
      "Previous_Fraudulent_Transactions": 7,
      "Number_of_Transactions_Last_24H": 15,
      "Transaction_Amount": 3000,
      "Transaction_Type": "ATM Withdrawal",
      "Device_Used": "Smartphone",
      "Location": "Los Angeles",
      "Time_of_Transaction": 20,
      "Fraudulent": 1
    },
    {
      "User_ID": "USR_4567",
      "Account_Age": 2,
      "Previous_Fraudulent_Transactions": 4,
      "Number_of_Transactions_Last_24H": 8,
      "Transaction_Amount": 1500,
      "Transaction_Type": "Online Purchase",
      "Device_Used": "Desktop",
      "Location": "New York",
      "Time_of_Transaction": 6,
      "Fraudulent": 1
    },
    {
      "User_ID": "USR_8890",
      "Account_Age": 4,
      "Previous_Fraudulent_Transactions": 2,
      "Number_of_Transactions_Last_24H": 10,
      "Transaction_Amount": 2500,
      "Tran

In [10]:
transactions = parsed if isinstance(parsed, list) else parsed.get("transactions", [])

# 🔍 DIAGNOSTIC
print("\n=== DIAGNOSTIC VALIDATION ===")
print(f"Champs attendus : {set(graph['nodes'])}")
if transactions:
    print(f"Champs reçus    : {set(transactions[0].keys())}")
    print(f"Valeur Fraudulent : {transactions[0].get('Fraudulent')} (type: {type(transactions[0].get('Fraudulent'))})")


=== DIAGNOSTIC VALIDATION ===
Champs attendus : {'Payment_Method', 'Time_of_Transaction', 'Location', 'Transaction_Type', 'Fraudulent', 'Account_Age', 'Device_Used', 'Number_of_Transactions_Last_24H'}
Champs reçus    : {'Time_of_Transaction', 'Location', 'Transaction_Amount', 'Transaction_Type', 'Previous_Fraudulent_Transactions', 'Fraudulent', 'Account_Age', 'User_ID', 'Device_Used', 'Number_of_Transactions_Last_24H'}
Valeur Fraudulent : 1 (type: <class 'int'>)
